# 🏋️ PulseFit: build a data warehouse in Snowflake
## Notebook 01 · Day 1: get the keys to Snowflake

| 01 Set up | 02 Bronze | 03 Silver & Gold | 04 Answer the CEO | 05 Superpowers |
|:-:|:-:|:-:|:-:|:-:|
| 📍 **you are here** | ⬜ | ⬜ | ⬜ | ⬜ |

### The story
It's **July 2026**. You have just joined **PulseFit**, a (fictional) chain of five fitness clubs in Sri Lanka,
as their **first data engineer**.

On your first morning the CEO stops at your desk with one question:

> **"Which members stopped visiting last quarter, and was it because of the price increase?"**

You heard this exact question in the lecture. Nobody at PulseFit can answer it, because the data lives in
**three systems that were never built to talk to each other**:

| Source system | What it knows | How the data reaches us |
|---|---|---|
| 📱 **Member App** | members, their home club and their plan | a CSV export of all members on the 1st of every month, plus a club list |
| 🚪 **Door scanners** | every check-in at every club | JSON events, one file per month |
| 💰 **Finance** | the price of each plan, and when prices changed | a price list exported from Excel as CSV |

**Your mission:** build PulseFit's first data warehouse in Snowflake and answer the CEO's question by the end of this session.

### The plan: five notebooks, one story
| # | Notebook | What you do | Key concepts |
|---|---|---|---|
| 01 | **Day 1: get the keys** | create compute and storage, switch on the data feeds | Snowflake architecture, virtual warehouses, databases & schemas, stages |
| 02 | **Bronze** | load the raw files exactly as they arrive | file formats, `COPY INTO`, JSON with `VARIANT`, rerunnable loads |
| 03 | **Silver & Gold** | clean, connect and model the data | ELT, de-duplication, SCD Type 2, star schema, data-quality tests |
| 04 | **Answer the CEO** | answer the question with SQL and charts | star-schema queries, history in action, pruning, caching, scaling |
| 05 | **Superpowers** | recover from mistakes, then wrap up | Time Travel, UNDROP, zero-copy cloning, access control, sharing, cost |

### ▶️ How to use these notebooks
* You **do not write any code**. Every cell is ready to run.
* Read a text cell ➜ run the code cell under it (**Ctrl + Enter** on Windows, **Cmd + Return** on Mac, or the ▶ button) ➜ look at the result.
* 👀 marks what to notice in a result. 🤔 marks questions to discuss with your neighbour (no code needed).
* Run cells **in order, top to bottom**. Every notebook is safe to run again from the top if something goes wrong.

## Step 1: Where am I? Meet Snowflake's three layers
Snowflake is built from **three layers that scale independently** (lecture: *"Three layers, scaled independently"*):

```
┌──────────────────────────────────────────────────────────────────┐
│ ☁️  CLOUD SERVICES: the brain                                     │
│     logins · security · metadata · query optimiser · transactions │
├──────────────────────────────────────────────────────────────────┤
│ ⚙️  COMPUTE: virtual warehouses (you create, start, stop, resize) │
│     PULSEFIT_WH (X-Small)  ·  ...as many others as you like       │
├──────────────────────────────────────────────────────────────────┤
│ 🗄️  STORAGE: one copy of all data, columnar and compressed        │
│     databases ▸ schemas ▸ tables, views, stages                   │
└──────────────────────────────────────────────────────────────────┘
```

The cell below asks Snowflake about **your session**. It needs no compute at all: the cloud-services layer
answers it from metadata.

In [ ]:
%%sql -r whoami
SELECT CURRENT_USER()      AS my_user,
       CURRENT_ROLE()      AS my_role,
       CURRENT_REGION()    AS cloud_region,
       CURRENT_VERSION()   AS snowflake_version;

👀 **What to notice**
* `MY_ROLE` is probably **ACCOUNTADMIN**, the most powerful role. That's fine for a personal trial.
  In a real company you would work with smaller roles; you'll create one in notebook 05.
* `CLOUD_REGION` shows which cloud provider and region your account runs in (for example `AWS_AP_SOUTH_1` = Amazon, Mumbai).
  Snowflake runs on AWS, Azure and Google Cloud.

## Step 2: Compute: create a virtual warehouse
A **virtual warehouse** is a cluster of compute that runs your queries. It holds no data: storage and compute
are separate (lecture: *"Compute: virtual warehouses"*).

| Size | Credits per hour (while running) |
|---|---|
| X-Small | 1 |
| Small | 2 |
| Medium | 4 |
| ... each size up | doubles |

You pay **per second while it runs** (with a 60-second minimum each time it starts). With **auto-suspend** it goes to
sleep when nobody is using it, and with **auto-resume** it wakes up by itself when the next query arrives.

In [ ]:
%%sql -r create_warehouse
CREATE WAREHOUSE IF NOT EXISTS PULSEFIT_WH
    WAREHOUSE_SIZE      = 'XSMALL'
    AUTO_SUSPEND        = 60
    AUTO_RESUME         = TRUE
    INITIALLY_SUSPENDED = TRUE
    COMMENT = 'PulseFit practical: compute for all our work';

-- From now on, run our queries on this warehouse
USE WAREHOUSE PULSEFIT_WH;

SHOW WAREHOUSES LIKE 'PULSEFIT_WH';

👀 **What to notice:** look at the `state`, `size` and `auto_suspend` columns.
Creating a new compute cluster took about a second, with no hardware to buy, install or configure.
`AUTO_SUSPEND = 60` means it sleeps (and stops costing money) after 60 seconds without queries.

## Step 3: Storage: databases, schemas and our three layers
Storage is organised as **database ▸ schema ▸ objects** (tables, views, stages, ...). We create two databases:

* **`PULSEFIT_SOURCE`**: plays the *outside world*. Its `LANDING` schema holds the files the source systems send us.
  In a real company this would be a cloud bucket (Amazon S3, Azure Blob Storage, Google Cloud Storage).
* **`PULSEFIT_DW`**: our **data warehouse**, with one schema per layer (lecture: *"The reference architecture"*):

```
 SOURCES                PULSEFIT_SOURCE             PULSEFIT_DW  (our data warehouse)
 📱 Member App    ─┐                         ┌──────────┐    ┌──────────┐    ┌──────────┐
 🚪 Door scanners ─┼─►  LANDING_ZONE  ─────► │  BRONZE  │ ─► │  SILVER  │ ─► │   GOLD   │ ─► 📊 answers
 💰 Finance       ─┘    (files, like S3)     │ raw copy │    │ cleaned, │    │   star   │
                                             │ as-is    │    │ joined,  │    │  schema  │
                                             │          │    │ history  │    │          │
                                             └──────────┘    └──────────┘    └──────────┘
                                              notebook 02     notebook 03     notebooks 03-04
```

Each layer reads only from the layer below it (lecture: *"Layered architecture"*), so you can rebuild or change a
layer without breaking the ones above.

In [ ]:
%%sql -r create_databases
-- The "outside world": where PulseFit's systems drop their export files
CREATE DATABASE IF NOT EXISTS PULSEFIT_SOURCE COMMENT = 'Pretend outside world: landing zone for source-system files';
CREATE SCHEMA   IF NOT EXISTS PULSEFIT_SOURCE.LANDING;

-- Our data warehouse, organised in three layers
CREATE DATABASE IF NOT EXISTS PULSEFIT_DW COMMENT = 'PulseFit data warehouse';
CREATE SCHEMA   IF NOT EXISTS PULSEFIT_DW.BRONZE COMMENT = 'Raw: an exact, replayable copy of the source data';
CREATE SCHEMA   IF NOT EXISTS PULSEFIT_DW.SILVER COMMENT = 'Cleaned, de-duplicated and integrated; history kept';
CREATE SCHEMA   IF NOT EXISTS PULSEFIT_DW.GOLD   COMMENT = 'Star schema: facts and dimensions, built for questions';

SHOW SCHEMAS IN DATABASE PULSEFIT_DW;

👀 **What to notice:** besides our three schemas, Snowflake adds `PUBLIC` (a default schema) and
`INFORMATION_SCHEMA` (read-only views that describe everything in the database: tables, columns, load history, ...).

## Step 4: A landing zone for files: the stage
A **stage** is a place where files live before they are loaded into tables.
* **External stage**: points at your own cloud bucket (S3, Azure, GCS).
* **Internal stage**: storage managed by Snowflake.

We create an internal stage called `LANDING_ZONE`. It plays the role of PulseFit's cloud bucket, where the three
source systems drop their export files.

In [ ]:
%%sql -r create_stage
CREATE STAGE IF NOT EXISTS PULSEFIT_SOURCE.LANDING.LANDING_ZONE
    DIRECTORY = (ENABLE = TRUE)
    COMMENT = 'Pretend cloud bucket: PulseFit systems drop their export files here';

-- What is in the landing zone right now? (nothing yet, so expect an empty result)
LIST @PULSEFIT_SOURCE.LANDING.LANDING_ZONE;

## Step 5: Switch on PulseFit's data feeds (simulator)
In real life the files would simply arrive. Here a **simulator** plays the part of PulseFit's three systems.
It invents realistic (fake) members and check-ins for **1 October 2025 to 30 June 2026** and drops the export files
into the landing zone. The data is generated the same way for everyone, so **your numbers will match your classmates'**.

> ⚠️ **You don't need to understand the simulator code.** Skim it if you're curious, then run the four cells
> below one after another and wait for each to finish (about 30 seconds in total).

A few things happened at PulseFit during those nine months, and the data "remembers" them:
* 🏃 Members joined, upgraded their plan, moved to another home club, and some stopped coming.
* 🔁 **Nimal Perera** (member `1001`) moved from the **Kandy** club to the **Colombo** club on **1 March 2026**
  (the same example as the lecture slide *"SCD Type 1 vs Type 2"*).
* 💸 On **15 March 2026** PulseFit announced that the **Standard** plan would go up from **LKR 8,000** to
  **LKR 9,500** per month from **1 April 2026**.
* 🐞 Like every real system, the sources are a bit messy. You'll find out how in notebook 03.

In [ ]:
%%sql -r simulator_1_reference_data
-- 🔧 SIMULATOR 1/4: clubs (Member App) and the price list (Finance)
USE SCHEMA PULSEFIT_SOURCE.LANDING;

CREATE OR REPLACE TEMPORARY TABLE SIM_CLUBS AS
SELECT column1 AS club_id, column2 AS club_code, column3 AS club_name,
       column4 AS city, column5 AS province, column6::DATE AS opened_on
FROM VALUES
    (1, 'CMB', 'PulseFit Colombo',    'Colombo',    'Western',       '2018-06-01'),
    (2, 'KDY', 'PulseFit Kandy',      'Kandy',      'Central',       '2019-03-15'),
    (3, 'GAL', 'PulseFit Galle',      'Galle',      'Southern',      '2020-01-10'),
    (4, 'NEG', 'PulseFit Negombo',    'Negombo',    'Western',       '2021-07-01'),
    (5, 'KUR', 'PulseFit Kurunegala', 'Kurunegala', 'North Western', '2022-11-20');

CREATE OR REPLACE TEMPORARY TABLE SIM_PRICE_LIST AS
SELECT column1 AS plan_code, column2 AS plan_name, column3 AS monthly_fee, column4 AS effective_from
FROM VALUES
    ('BASIC',    'Basic (off-peak)',    '5,500',  '01/01/2025'),
    ('STANDARD', 'Standard',            '8,000',  '01/01/2025'),
    ('PREMIUM',  'Premium (all clubs)', '14,000', '01/01/2025'),
    ('STUDENT',  'Student',             '4,500',  '01/01/2025'),
    ('STANDARD', 'Standard',            '9,500',  '01/04/2026');

SELECT * FROM SIM_CLUBS;

In [ ]:
%%sql -r simulator_2_members
-- 🔧 SIMULATOR 2/4: 2,400 members and how they behave, plus a members export on the 1st of every month.
-- HASH(...) gives "random" numbers that are identical for everyone, so all students get the same data.
CREATE OR REPLACE TEMPORARY TABLE SIM_MEMBERS AS
WITH ids AS (
    SELECT 1000 + ROW_NUMBER() OVER (ORDER BY SEQ4()) AS member_id
    FROM TABLE(GENERATOR(ROWCOUNT => 2400))
), r AS (
    SELECT member_id,
           ABS(HASH(member_id, 'club'))   % 10000 / 10000 AS r_club,
           ABS(HASH(member_id, 'plan'))   % 10000 / 10000 AS r_plan,
           ABS(HASH(member_id, 'join'))   % 10000 / 10000 AS r_join,
           ABS(HASH(member_id, 'freq'))   % 10000 / 10000 AS r_freq,
           ABS(HASH(member_id, 'churn'))  % 10000 / 10000 AS r_churn,
           ABS(HASH(member_id, 'churnd')) % 10000 / 10000 AS r_churn_day,
           ABS(HASH(member_id, 'price'))  % 10000 / 10000 AS r_price,
           ABS(HASH(member_id, 'priced')) % 10000 / 10000 AS r_price_day,
           ABS(HASH(member_id, 'cancel')) % 10000 / 10000 AS r_cancel,
           ABS(HASH(member_id, 'move'))   % 10000 / 10000 AS r_move,
           ABS(HASH(member_id, 'moved'))  % 10000 / 10000 AS r_move_day,
           ABS(HASH(member_id, 'upg'))    % 10000 / 10000 AS r_upgrade,
           ABS(HASH(member_id, 'upgd'))   % 10000 / 10000 AS r_upgrade_day,
           ABS(HASH(member_id, 'email'))  % 10000 / 10000 AS r_email,
           ABS(HASH(member_id, 'fn')) AS h_fn,
           ABS(HASH(member_id, 'ln')) AS h_ln,
           ABS(HASH(member_id, 'by')) AS h_by
    FROM ids
), m1 AS (
    SELECT r.*,
           ARRAY_CONSTRUCT('Nimal','Kamal','Sunil','Amal','Saman','Ruwan','Chamara','Dilshan','Kasun','Tharindu',
                           'Nuwan','Lahiru','Isuru','Pradeep','Ashan','Sachini','Nadeesha','Dilini','Kavindi','Hiruni',
                           'Tharushi','Nethmi','Ishara','Malsha','Anjali','Priyanka','Fathima','Ayesha','Rizwan','Arjun',
                           'Kumaran','Ravi','Shanika','Gayani','Madushani','Chathura','Supun','Yasiru','Hasini','Senuri')[h_fn % 40]::STRING AS first_name,
           ARRAY_CONSTRUCT('Perera','Fernando','Silva','Jayasinghe','Bandara','Wickramasinghe','Gunawardena','Dissanayake',
                           'Herath','Wijesinghe','Ranasinghe','Kumara','Rathnayake','Mendis','De Silva','Weerasinghe',
                           'Karunaratne','Amarasinghe','Nanayakkara','Ekanayake','Samarawickrama','Hettiarachchi',
                           'Abeysekera','Jayawardena','Liyanage','Gamage','Seneviratne','Munasinghe','Peiris','Cooray')[h_ln % 30]::STRING AS last_name,
           CASE WHEN r_club < 0.36 THEN 1 WHEN r_club < 0.56 THEN 2 WHEN r_club < 0.72 THEN 3 WHEN r_club < 0.87 THEN 4 ELSE 5 END AS home_club_id,
           CASE WHEN r_plan < 0.20 THEN 'BASIC' WHEN r_plan < 0.65 THEN 'STANDARD' WHEN r_plan < 0.80 THEN 'PREMIUM' ELSE 'STUDENT' END AS plan_code,
           DATEADD(day, FLOOR(r_join * 1247), '2023-01-01'::DATE) AS joined_on
    FROM r
), m2 AS (
    SELECT m1.*,
           IFF(plan_code = 'STUDENT', 2000 + h_by % 8, 1965 + h_by % 38) AS birth_year,
           1.2 + r_freq * 3.3 + IFF(plan_code = 'PREMIUM', 0.5, 0) AS visits_per_week,
           IFF(r_upgrade < 0.08 AND plan_code IN ('BASIC', 'STANDARD'),
               DATEADD(day, FLOOR(r_upgrade_day * 120), '2025-10-20'::DATE), NULL) AS upgrade_date,
           DECODE(plan_code, 'BASIC', 'STANDARD', 'STANDARD', 'PREMIUM') AS upgrade_plan,
           IFF(r_move < 0.06, DATEADD(day, FLOOR(r_move_day * 240), '2025-10-15'::DATE), NULL) AS move_date,
           MOD(home_club_id + FLOOR(r_move_day * 1000) % 4, 5) + 1 AS move_club_id
    FROM m1
), m3 AS (
    SELECT m2.*,
           IFF(upgrade_date <= '2026-04-01', upgrade_plan, plan_code) AS plan_on_price_day,
           IFF(r_churn < 0.12, DATEADD(day, FLOOR(r_churn_day * 273), '2025-10-01'::DATE), NULL) AS natural_churn_date
    FROM m2
), m4 AS (
    -- one in three Standard members stops coming after the price rise is announced (15-31 March 2026)
    SELECT m3.*,
           IFF(plan_on_price_day = 'STANDARD' AND r_price < 0.33,
               DATEADD(day, FLOOR(r_price_day * 17), '2026-03-15'::DATE), NULL) AS price_churn_date
    FROM m3
), m5 AS (
    SELECT m4.*,
           NULLIF(LEAST(COALESCE(IFF(natural_churn_date > DATEADD(day, 21, joined_on), natural_churn_date, NULL), '9999-12-31'::DATE),
                        COALESCE(IFF(price_churn_date   > DATEADD(day, 21, joined_on), price_churn_date,   NULL), '9999-12-31'::DATE)),
                  '9999-12-31'::DATE) AS lapse_date
    FROM m4
)
SELECT member_id,
       'CARD-' || LPAD((member_id * 37) % 10007, 5, '0') AS card_id,
       IFF(member_id = 1001, 'Nimal',  first_name) AS first_name,
       IFF(member_id = 1001, 'Perera', last_name)  AS last_name,
       IFF(r_email < 0.10 AND member_id <> 1001,
           ' ' || first_name || '.' || REPLACE(last_name, ' ', '') || (member_id % 97) || '@Example.com ',
           LOWER(IFF(member_id = 1001, 'Nimal', first_name) || '.'
                 || REPLACE(IFF(member_id = 1001, 'Perera', last_name), ' ', '') || (member_id % 97)) || '@example.com') AS email,
       birth_year,
       IFF(member_id = 1001, 2, home_club_id)                AS home_club_id,
       IFF(member_id = 1001, 'PREMIUM', plan_code)           AS plan_code,
       IFF(member_id = 1001, '2024-02-10'::DATE, joined_on)  AS joined_on,
       IFF(member_id = 1001, 3, visits_per_week)             AS visits_per_week,
       IFF(member_id = 1001, NULL, upgrade_date)             AS upgrade_date,
       upgrade_plan,
       IFF(member_id = 1001, '2026-03-01'::DATE, move_date)  AS move_date,
       IFF(member_id = 1001, 1, move_club_id)                AS move_club_id,
       IFF(member_id = 1001, NULL, lapse_date)               AS lapse_date,
       IFF(member_id <> 1001 AND lapse_date IS NOT NULL AND r_cancel < 0.5,
           DATEADD(day, 30 + FLOOR(r_cancel * 60), lapse_date), NULL) AS cancel_date
FROM m5;

-- What the Member App exports on the 1st of each month: every member as they look on that day
CREATE OR REPLACE TEMPORARY TABLE SIM_MEMBER_SNAPSHOTS AS
SELECT s.snapshot_date, m.member_id, m.card_id, m.first_name, m.last_name, m.email, m.birth_year,
       IFF(m.move_date    <= s.snapshot_date, m.move_club_id, m.home_club_id) AS home_club_id,
       IFF(m.upgrade_date <= s.snapshot_date, m.upgrade_plan, m.plan_code)    AS plan_code,
       IFF(m.cancel_date  <= s.snapshot_date, 'CANCELLED', 'ACTIVE')          AS status,
       m.joined_on
FROM SIM_MEMBERS m
JOIN (SELECT DATEADD(month, ROW_NUMBER() OVER (ORDER BY SEQ4()) - 1, '2025-10-01'::DATE) AS snapshot_date
      FROM TABLE(GENERATOR(ROWCOUNT => 10))) s
  ON m.joined_on < s.snapshot_date;

SELECT COUNT(DISTINCT member_id) AS members, COUNT(*) AS rows_in_all_monthly_exports FROM SIM_MEMBER_SNAPSHOTS;

In [ ]:
%%sql -r simulator_3_door_scanners
-- 🔧 SIMULATOR 3/4: every check-in at every club, as JSON events from the door scanners.
-- Scanners send times in UTC and identify people by their membership card, not by member id.
CREATE OR REPLACE TEMPORARY TABLE SIM_CHECKINS AS
WITH days AS (
    SELECT DATEADD(day, ROW_NUMBER() OVER (ORDER BY SEQ4()) - 1, '2025-10-01'::DATE) AS visit_date
    FROM TABLE(GENERATOR(ROWCOUNT => 273))
), cand AS (
    SELECT m.member_id, m.card_id, d.visit_date, m.visits_per_week,
           IFF(m.move_date <= d.visit_date, m.move_club_id, m.home_club_id) AS home_club_today,
           ABS(HASH(m.member_id, d.visit_date, 'v')) % 10000 / 10000 AS r_visit,
           ABS(HASH(m.member_id, d.visit_date, 'c')) % 10000 / 10000 AS r_club,
           ABS(HASH(m.member_id, d.visit_date, 't')) % 10000 / 10000 AS r_time,
           ABS(HASH(m.member_id, d.visit_date, 'm')) % 60            AS minute_of_hour,
           ABS(HASH(m.member_id, d.visit_date, 'd')) % 10000 / 10000 AS r_duration,
           ABS(HASH(m.member_id, d.visit_date, 'a1')) % 100 AS a1,
           ABS(HASH(m.member_id, d.visit_date, 'a2')) % 100 AS a2,
           ABS(HASH(m.member_id, d.visit_date, 'a3')) % 100 AS a3,
           ABS(HASH(m.member_id, d.visit_date, 'a4')) % 100 AS a4,
           ABS(HASH(m.member_id, d.visit_date, 'a5')) % 100 AS a5,
           ABS(HASH(m.member_id, d.visit_date, 'dup')) % 1000 AS r_dup
    FROM SIM_MEMBERS m
    JOIN days d ON d.visit_date >= m.joined_on AND d.visit_date < COALESCE(m.lapse_date, '9999-12-31'::DATE)
), visits AS (
    SELECT *,
           IFF(member_id = 1001 OR r_club < 0.9, home_club_today,
               MOD(home_club_today + FLOOR(r_club * 1000) % 4, 5) + 1) AS club_id,
           DATEADD(minute,
                   IFF(r_time < 0.42, 5 + FLOOR(r_time / 0.42 * 5), 16 + FLOOR((r_time - 0.42) / 0.58 * 6)) * 60 + minute_of_hour,
                   visit_date::TIMESTAMP_NTZ) AS local_ts
    FROM cand
    WHERE (member_id <> 1001 AND r_visit < visits_per_week / 7 * IFF(DAYOFWEEKISO(visit_date) = 7, 0.6, 1.0))
       OR (member_id = 1001 AND DAYOFWEEKISO(visit_date) IN (1, 3, 5)
           AND visit_date NOT BETWEEN '2025-12-22' AND '2026-01-04')
), events AS (
    SELECT v.r_dup,
           OBJECT_CONSTRUCT(
               'event_id',   'evt_' || SUBSTR(SHA1(v.member_id || '|' || v.visit_date), 1, 20),
               'event_type', 'CHECK_IN',
               'event_ts',   TO_VARCHAR(DATEADD(minute, -330, v.local_ts), 'YYYY-MM-DD"T"HH24:MI:SS"Z"'),
               'card_id',    v.card_id,
               'device',     OBJECT_CONSTRUCT('club_code', c.club_code,
                                              'scanner_id', c.club_code || '-DOOR-' || (1 + v.minute_of_hour % 2)),
               'session',    OBJECT_CONSTRUCT(
                                 'duration_min', 30 + FLOOR(v.r_duration * 90),
                                 'activities', IFF(a1 >= 55 AND a2 >= 50 AND a3 >= 15 AND a4 >= 12 AND a5 >= 8,
                                                   ARRAY_CONSTRUCT('cardio'),
                                                   ARRAY_CONSTRUCT_COMPACT(
                                                       IFF(a1 < 55, 'cardio', NULL), IFF(a2 < 50, 'weights', NULL),
                                                       IFF(a3 < 15, 'yoga', NULL),   IFF(a4 < 12, 'spin', NULL),
                                                       IFF(a5 < 8, 'swimming', NULL))))
           ) AS payload
    FROM visits v
    JOIN SIM_CLUBS c ON c.club_id = v.club_id
), guests AS (
    SELECT OBJECT_CONSTRUCT(
               'event_id',   'evt_' || SUBSTR(SHA1('guest|' || n), 1, 20),
               'event_type', 'CHECK_IN',
               'event_ts',   TO_VARCHAR(DATEADD(minute, ABS(HASH(n, 'gt')) % 900 + 30,
                                                DATEADD(day, ABS(HASH(n, 'gd')) % 273, '2025-10-01'::TIMESTAMP_NTZ)),
                                        'YYYY-MM-DD"T"HH24:MI:SS"Z"'),
               'card_id',    'GUEST-' || LPAD(n, 4, '0'),
               'device',     OBJECT_CONSTRUCT('club_code', ARRAY_CONSTRUCT('CMB','KDY','GAL','NEG','KUR')[ABS(HASH(n, 'gc')) % 5]::STRING,
                                              'scanner_id', 'GUEST-DESK'),
               'session',    OBJECT_CONSTRUCT('duration_min', 60, 'activities', ARRAY_CONSTRUCT('cardio'))
           ) AS payload
    FROM (SELECT ROW_NUMBER() OVER (ORDER BY SEQ4()) AS n FROM TABLE(GENERATOR(ROWCOUNT => 350)))
)
SELECT payload FROM events
UNION ALL SELECT payload FROM events WHERE r_dup < 12
UNION ALL SELECT payload FROM guests;

SELECT COUNT(*) AS scanner_events FROM SIM_CHECKINS;

In [ ]:
%%sql -r simulator_4_export_files
-- 🔧 SIMULATOR 4/4: each system drops its export files into the landing zone
REMOVE @PULSEFIT_SOURCE.LANDING.LANDING_ZONE;

COPY INTO @PULSEFIT_SOURCE.LANDING.LANDING_ZONE/member_app/clubs/clubs.csv
FROM (SELECT club_id, club_code, club_name, city, province, opened_on FROM SIM_CLUBS ORDER BY club_id)
FILE_FORMAT = (TYPE = CSV COMPRESSION = NONE FIELD_OPTIONALLY_ENCLOSED_BY = '"' NULL_IF = ())
HEADER = TRUE SINGLE = TRUE OVERWRITE = TRUE;

COPY INTO @PULSEFIT_SOURCE.LANDING.LANDING_ZONE/finance/price_list.csv
FROM (SELECT plan_code AS "Plan Code", plan_name AS "Plan Name",
             monthly_fee AS "Monthly Fee (LKR)", effective_from AS "Effective From"
      FROM SIM_PRICE_LIST)
FILE_FORMAT = (TYPE = CSV COMPRESSION = NONE FIELD_OPTIONALLY_ENCLOSED_BY = '"' NULL_IF = ())
HEADER = TRUE SINGLE = TRUE OVERWRITE = TRUE;

COPY INTO @PULSEFIT_SOURCE.LANDING.LANDING_ZONE/member_app/members/
FROM (SELECT snapshot_date, member_id, card_id, first_name, last_name, email, birth_year,
             home_club_id, plan_code, status, joined_on
      FROM SIM_MEMBER_SNAPSHOTS)
PARTITION BY ('members_' || TO_VARCHAR(snapshot_date, 'YYYY-MM-DD'))
FILE_FORMAT = (TYPE = CSV COMPRESSION = NONE FIELD_OPTIONALLY_ENCLOSED_BY = '"' NULL_IF = ())
HEADER = TRUE;

COPY INTO @PULSEFIT_SOURCE.LANDING.LANDING_ZONE/door_scanners/
FROM (SELECT payload FROM SIM_CHECKINS)
PARTITION BY ('checkins_' || SUBSTR(payload:event_ts::STRING, 1, 7))
FILE_FORMAT = (TYPE = JSON COMPRESSION = GZIP);

-- Summarise what arrived, using the stage's directory table (a queryable list of its files)
ALTER STAGE PULSEFIT_SOURCE.LANDING.LANDING_ZONE REFRESH;

SELECT SPLIT_PART(relative_path, '/', 1)  AS source_system,
       COUNT(*)                            AS files,
       ROUND(SUM(size) / 1024 / 1024, 2)   AS megabytes
FROM DIRECTORY(@PULSEFIT_SOURCE.LANDING.LANDING_ZONE)
GROUP BY 1
ORDER BY 1;

👀 **What to notice:** the landing zone now holds about two dozen files from three systems:
`door_scanners` (compressed JSON, one file per month), `finance` (one CSV) and `member_app`
(a club list plus ten monthly member exports).
PulseFit's data feeds are switched on. 🎉

## ✅ What you did
* Met Snowflake's three layers: **cloud services**, **compute** and **storage**.
* Created a **virtual warehouse** that suspends itself when idle, so you only pay while it works.
* Created **databases** and **schemas** for a layered warehouse: `BRONZE` ▸ `SILVER` ▸ `GOLD`.
* Created a **stage** and filled it with export files from three source systems.

## 🤔 Think about it
1. You just created a compute cluster in about a second. In a traditional on-premise warehouse, what would
   "we need more compute" have involved, and how long would it take?
2. Why keep the raw files in a landing zone at all, instead of loading them straight into clean tables?

## ➡️ Next
Open **`02_Bronze_Load_Raw_Data`**. You'll load these files into the warehouse.